# ex_factors 单股对齐验证

目标：验证磁盘已有 ex_factors 与 rqdatac 实时拉取是否一致，确认可以安全增量更新。

测试股：600519.XSHG（贵州茅台，上市久、除权事件多，适合做对齐基准）

In [ ]:
import os, pandas as pd, rqdatac
from pathlib import Path

# 从 .env 加载凭据
env_path = Path('.env')
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ[k.strip()] = v.strip()

rqdatac.init(os.environ.get('RQDATAC_USERID'), os.environ.get('RQDATAC_PASSWORD'))
print(f'rqdatac ready, latest trading date: {rqdatac.get_latest_trading_date()}')

In [ ]:
STOCK = '600519.XSHG'
DATA_ROOT = Path('/nfs/ofs-prediction/peterzhenglinpeng')
EX_DIR = DATA_ROOT / 'market-data' / 'daily' / 'stock-ex-factors'

# 读磁盘已有数据
disk_path = EX_DIR / f'{STOCK}.parquet'
disk = pd.read_parquet(disk_path)
print(f'磁盘 ex_factors: {disk.shape}, range={disk.index.min().date()} ~ {disk.index.max().date()}')
print(f'columns: {disk.columns.tolist()}')
print()
print('磁盘数据（全部）：')
disk

In [ ]:
# 用相同区间从 rqdatac 拉取
start = str(disk.index.min().date())
end = str(disk.index.max().date())

live = rqdatac.get_ex_factor([STOCK], start_date=start, end_date=end)
if live is None or len(live) == 0:
    print('⚠️ rqdatac 返回空——可能区间无除权事件')
else:
    live = live.reset_index()
    if 'order_book_id' in live.columns:
        live = live.drop(columns=['order_book_id'])
    live = live.set_index('ex_date').sort_index()
    print(f'rqdatac 拉取: {live.shape}')
    print(live)

In [ ]:
# 逐值对齐比对
if live is not None and len(live) > 0:
    common_idx = disk.index.intersection(live.index)
    print(f'重叠日期数: {len(common_idx)}')
    
    compare_cols = [c for c in ['ex_cum_factor', 'ex_factor'] if c in disk.columns and c in live.columns]
    print(f'比对列: {compare_cols}')
    
    all_ok = True
    for col in compare_cols:
        a = disk.loc[common_idx, col].values.astype(float)
        b = live.loc[common_idx, col].values.astype(float)
        both_valid = ~pd.isna(a) & ~pd.isna(b)
        max_diff = float(abs(a[both_valid] - b[both_valid]).max()) if both_valid.any() else 0
        match = max_diff < 1e-6
        print(f'  {col}: max_diff={max_diff:.2e} → {"✅ 一致" if match else "❌ 不一致"}')
        if not match:
            all_ok = False
    
    if all_ok:
        print('\n✅ 对齐验证通过：磁盘数据与 rqdatac 逐值一致，可安全增量更新')
    else:
        print('\n❌ 对齐验证失败：需要排查口径差异')
else:
    print('跳过比对（live 为空）')

In [ ]:
# 检查磁盘截止日之后是否有新除权事件
disk_end = disk.index.max()
today = pd.Timestamp(rqdatac.get_latest_trading_date())

print(f'磁盘截止: {disk_end.date()}')
print(f'最新交易日: {today.date()}')
print(f'差距: {(today - disk_end).days} 天')
print()

# 拉磁盘截止日之后的数据
new_start = str((disk_end + pd.Timedelta(days=1)).date())
new_end = str(today.date())

new_data = rqdatac.get_ex_factor([STOCK], start_date=new_start, end_date=new_end)
if new_data is None or len(new_data) == 0:
    print(f'✅ {new_start} ~ {new_end} 无新除权事件，磁盘已是最新')
else:
    new_data = new_data.reset_index()
    if 'order_book_id' in new_data.columns:
        new_data = new_data.drop(columns=['order_book_id'])
    new_data = new_data.set_index('ex_date').sort_index()
    print(f'⚠️ 发现 {len(new_data)} 条新除权事件：')
    print(new_data)
    
    # 模拟 append + dedup
    combined = pd.concat([disk, new_data])
    combined = combined[~combined.index.duplicated(keep='last')].sort_index()
    print(f'\n合并后: {combined.shape}, range={combined.index.min().date()} ~ {combined.index.max().date()}')
    print(combined.tail(5))

In [ ]:
# 抽样多只股票做批量对齐（验证 ex_factors 整体一致性）
import numpy as np

sample_stocks = ['600519.XSHG', '000001.XSHE', '300750.XSHE', '601318.XSHG', '000858.XSHE']
results = []

for stock in sample_stocks:
    path = EX_DIR / f'{stock}.parquet'
    if not path.exists():
        results.append({'stock': stock, 'status': '文件不存在'})
        continue
    
    d = pd.read_parquet(path)
    if len(d) == 0:
        results.append({'stock': stock, 'status': '空文件'})
        continue
    
    # 拉相同区间
    l = rqdatac.get_ex_factor([stock], start_date=str(d.index.min().date()), end_date=str(d.index.max().date()))
    if l is None or len(l) == 0:
        results.append({'stock': stock, 'status': 'rqdatac空', 'disk_rows': len(d)})
        continue
    
    l = l.reset_index()
    if 'order_book_id' in l.columns:
        l = l.drop(columns=['order_book_id'])
    l = l.set_index('ex_date').sort_index()
    
    ci = d.index.intersection(l.index)
    if len(ci) == 0:
        results.append({'stock': stock, 'status': '无重叠', 'disk_rows': len(d), 'live_rows': len(l)})
        continue
    
    a = d.loc[ci, 'ex_cum_factor'].values.astype(float)
    b = l.loc[ci, 'ex_cum_factor'].values.astype(float)
    both = ~pd.isna(a) & ~pd.isna(b)
    maxd = float(abs(a[both] - b[both]).max()) if both.any() else np.nan
    
    results.append({
        'stock': stock,
        'disk_rows': len(d),
        'live_rows': len(l),
        'overlap': len(ci),
        'max_diff': maxd,
        'status': '✅' if maxd < 1e-6 else '❌'
    })

pd.DataFrame(results)

In [ ]:
# 结论
print('=== 验证结论 ===')
print('如果上面全部 ✅，说明：')
print('  1. 磁盘 ex_factors 与 rqdatac 口径完全一致')
print('  2. 可以安全用 ex_factors.py 增量更新（append + dedup）')
print('  3. 更新后 raw_ohlcv 的复权计算会自动使用最新的 cum_factor')
print()
print('下一步：运行 data_fetching/ex_factors.py 更新复权因子')

## 验证记录（2026-06-13）

| 股票 | 磁盘行数 | 重叠数 | maxΔ | 状态 | 截止日 | 新增事件 |
|------|---------|--------|------|------|--------|----------|
| 600519.XSHG | 29 | 29 | 0.00e+00 | ✅ | 2025-12-19 | 0 |
| 000001.XSHE | 30 | 30 | 0.00e+00 | ✅ | 2025-10-15 | 1 |
| 300750.XSHE | 10 | 10 | 0.00e+00 | ✅ | 2026-04-22 | 0 |
| 601318.XSHG | 37 | 37 | 0.00e+00 | ✅ | 2025-10-24 | 1 |
| 000858.XSHE | 29 | 29 | 0.00e+00 | ✅ | 2025-12-18 | 0 |

**结论：全部通过，max_diff = 0，可安全增量更新。**

下一步：`python data_fetching/ex_factors.py` 更新全量复权因子

In [ ]:
import pandas as pd
from dquant import data as ddata
rq = pd.read_parquet(f'/nfs/ofs-prediction/peterzhenglinpeng/market-data/daily/stock-ohlcv/{code}.parquet')
rq.index = pd.to_datetime(rq.index)
rq_start, rq_end = rq.index.min().date(), rq.index.max().date()
dq = ddata.get_price(order_book_ids=[code], start_date='20050101', end_date='20261231',
                             frequency='1d', source='rq')